In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.8 Qubits, the Bloch Sphere, and a First Taste of Entanglement

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.8",
    title="Qubits, the Bloch Sphere, and a First Taste of Entanglement",
    blurb="A picture, and then a surprise. Every state of a single qubit is a point "
    "on a sphere, every evolution a rotation — the geometry that makes all of "
    "Movement I visible at once. Then we put two qubits together and meet the "
    "phenomenon with no classical shadow: entanglement, where the whole is sharp but "
    "each part is blurred, and the information lives entirely in the correlations "
    "between them.",
    difficulty="advanced",
    estimate="160–195 min",
)

## Notebook overview

This is the capstone of Movement I, and it does two things. First it gives the qubit its **geometry**.
Everything we have built for a two-level system — superposition, measurement, uncertainty, dynamics —
turns out to fit on the surface of a single sphere, and once that picture is in hand, the whole
movement can be seen at a glance. A qubit's physical state, a *ray* in $\mathbb{C}^2$, is a **point**
on the **Bloch sphere**; its coordinates are the Pauli expectation values; orthogonal states sit at
opposite poles; and a unitary operation is a **rotation** — the Larmor precession of [§6.7](time-evolution.ipynb) made literal,
a Bloch vector spinning about the field axis.

Then the notebook takes the first real step beyond a single particle, and meets the one phenomenon
that has no classical shadow at all. Two qubits live in the **tensor product** $\mathbb{C}^2\otimes
\mathbb{C}^2$, and most of the states there are *not* products of individual states: they are
**entangled**. The canonical example is the Bell state $(|00\rangle+|11\rangle)/\sqrt2$, and its
signature is startling — the *whole* is a perfectly definite (pure) state, yet *each part*, taken
alone, is maximally uncertain. The information is not in the pieces; it is entirely in the
**correlations** between them. We diagnose this precisely (the Schmidt rank, the reduced state's
mixedness) and end on the puzzle that breaks classical realism: correlations perfect in two
*incompatible* bases, which no pre-assigned values could ever reproduce.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the Bloch vector from Pauli expectation values, `numpy.kron`
for the tensor product, `numpy.linalg.svd` for the Schmidt decomposition, a reshape-and-contract for
the reduced density matrix, and `numpy.linalg.eigvalsh` for its spectrum.

> **A boundary, stated up front.** Entanglement forces us to meet a new object — the **reduced density
> matrix** $\rho_A=\mathrm{Tr}_B|\psi\rangle\langle\psi|$, the state of *part* of an entangled pair,
> together with its **purity** $\mathrm{Tr}(\rho^2)$ and **entanglement entropy** $S=-\mathrm{Tr}(\rho
> \log_2\rho)$ (the 5.x entropy of the reduced state). We use it here only as the diagnostic of
> entanglement; the *general* density-matrix formalism — mixed states from classical ignorance, the
> full theory — is [§6.26](density-matrix.ipynb). Likewise the **Bell correlations** here are a *taste*; the quantitative
> Bell-inequality violation is [§6.25](bell-inequality.ipynb).
>
> **How to read the checks.** Each exercise closes with a `validate` call: the Bloch map and unit
> purity of a pure state; global-phase invariance; orthogonal states antipodal and precession a
> rotation; the tensor-product structure; the Bell state's Schmidt rank 2 versus a product's rank 1;
> the Bell reduced state maximally mixed (purity $\tfrac12$, $S=1$ bit); and the perfect Bell
> correlations with random marginals. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*.
>
> **Conventions and scope.** A qubit is parametrized as $|\psi\rangle=\cos\tfrac\theta2|0\rangle+
> e^{i\varphi}\sin\tfrac\theta2|1\rangle$; in `numpy.kron(a, b)` the first factor is qubit $A$, so
> $|q_Aq_B\rangle$ is index $2q_A+q_B$. The Pauli matrices are those of [§6.6](pauli-uncertainty.ipynb). Mixed states (inside the
> sphere) and the general density operator are [§6.26](density-matrix.ipynb); Bell's inequality is [§6.25](bell-inequality.ipynb); wave mechanics (the
> same formalism in infinite dimensions) opens Movement II in [§6.9](position-representation.ipynb). See Nielsen & Chuang; Sakurai &
> Napolitano; and Notebooks [§6.1](complex-vector-spaces.ipynb) (rays, global phase), [§6.6](pauli-uncertainty.ipynb) (Pauli expectations), [§6.7](time-evolution.ipynb) (precession), 5.x
> (entropy).

## Theory in brief

### The Bloch sphere

Any qubit state, with its global phase removed, is

```{math}
:label: eq-bloch
|\psi\rangle=\cos\tfrac\theta2|0\rangle+e^{i\varphi}\sin\tfrac\theta2|1\rangle\ \mapsto\ \mathbf b=(\langle\sigma_x\rangle,\langle\sigma_y\rangle,\langle\sigma_z\rangle)=(\sin\theta\cos\varphi,\ \sin\theta\sin\varphi,\ \cos\theta) ,
```

a **point** on the unit sphere whose Cartesian coordinates are the **Pauli expectation values** (the
Bloch vector). The global phase of [§6.1](complex-vector-spaces.ipynb) is exactly what is quotiented away — the ray becomes a point —
and a **pure** state has $|\mathbf b|=1$. Poles: $|0\rangle$ (north), $|1\rangle$ (south); equator:
$|{\pm}\rangle$, $|{\pm}i\rangle$.

### Geometry of states and evolution

The Bloch map turns Hilbert-space relations into spherical geometry: a short computation in the
half-angle parametrization gives $|\langle\phi|\psi\rangle|^2=\tfrac12(1+\mathbf b_\phi\cdot\mathbf b_\psi)$
for pure states (Nielsen & Chuang develop the full dictionary), so overlaps become angles and
overlap-preserving unitaries become rotations:

```{math}
:label: eq-bloch-geometry
\langle\phi|\psi\rangle=0\ \Longleftrightarrow\ \mathbf b_\phi=-\mathbf b_\psi\ (\text{antipodal}),\qquad U\ \text{unitary}\ \Longleftrightarrow\ \text{rotation of the sphere} .
```

**Orthogonal** states are **antipodal** (note the factor of two between the Hilbert-space angle and
the Bloch angle — the spin-$\tfrac12$ double cover), a **unitary** is a **rotation**, and the Larmor
precession of [§6.7](time-evolution.ipynb) is literally the Bloch vector rotating about the field at the Larmor frequency.
(Mixed states, [§6.26](density-matrix.ipynb), live *inside* the sphere.)

### The tensor product: composite systems

Two qubits live in the **tensor product**,

```{math}
:label: eq-tensor
\mathbb{C}^2\otimes\mathbb{C}^2=\mathbb{C}^4,\quad \{|00\rangle,|01\rangle,|10\rangle,|11\rangle\},\qquad |a\rangle\otimes|b\rangle\ (\texttt{numpy.kron}),\quad A\otimes I\ \text{acts on qubit 1} .
```

A **product** (separable) state is $|a\rangle\otimes|b\rangle$; a single-qubit observable is $A\otimes
I$. This is how quantum mechanics describes more than one particle.

### Entanglement

Some two-qubit states are **not** products,

```{math}
:label: eq-entanglement
|\Phi^+\rangle=\tfrac{1}{\sqrt2}\big(|00\rangle+|11\rangle\big)\ \ne\ |a\rangle\otimes|b\rangle\ \text{for any }a,b .
```

The **Schmidt decomposition** diagnoses it: reshape the 4-vector to a $2\times2$ matrix and take its
singular values (`numpy.linalg.svd`). A product state has **Schmidt rank 1** (one nonzero singular
value); an **entangled** state has rank $\ge2$. The Bell state has rank 2 with equal coefficients.

### The signature: the reduced state is mixed

For an entangled pure state, the **reduced density matrix** of one qubit is **mixed**,

```{math}
:label: eq-reduced
\rho_A=\mathrm{Tr}_B|\psi\rangle\langle\psi|,\qquad \text{Bell:}\ \rho_A=\tfrac12 I\ (\text{purity }\mathrm{Tr}\rho_A^2=\tfrac12,\ S=-\mathrm{Tr}\rho_A\log_2\rho_A=1\ \text{bit}) ,
```

*maximally* mixed — even though the whole is pure. A product state instead gives a pure reduced state
(purity 1, $S=0$). The whole is sharp, each part maximally uncertain: the information is in the
correlations, not the pieces. (This is the first density matrix; the general theory is [§6.26](density-matrix.ipynb).)

### Bell correlations — the taste

In the Bell state the two qubits are **perfectly correlated** in two *incompatible* bases while each
alone is random,

```{math}
:label: eq-bell-taste
\langle\sigma_z\otimes\sigma_z\rangle=+1,\quad \langle\sigma_x\otimes\sigma_x\rangle=+1,\qquad \langle\sigma_z\otimes I\rangle=\langle I\otimes\sigma_z\rangle=0 .
```

Correlations this strong in incompatible bases cannot come from pre-assigned (local hidden-variable)
values — the puzzle Bell's inequality ([§6.25](bell-inequality.ipynb)) makes quantitative.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers the 3d projection)

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: a qubit is |ψ⟩=cos(θ/2)|0⟩+e^{iφ}sin(θ/2)|1⟩; the Bloch vector is the triple
# of Pauli expectation values (numpy.vdot). For two qubits, numpy.kron(a, b) puts qubit A first, so
# |q_A q_B⟩ is index 2·q_A + q_B; A⊗I = numpy.kron(A, I) acts on qubit A. ℏ=1.

SIGMA_X = np.array([[0, 1], [1, 0]], dtype=complex)
SIGMA_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
SIGMA_Z = np.array([[1, 0], [0, -1]], dtype=complex)
ID2 = np.eye(2, dtype=complex)


def qubit(theta, phi=0.0):
    """A qubit state $\\cos\\tfrac\\theta2|0\\rangle+e^{i\\varphi}\\sin\\tfrac\\theta2|1\\rangle$ {eq}`eq-bloch`.

    The half-angle parametrization (the spin-$\\tfrac12$ signature of §6.4) places the state at the point
    $(\\theta,\\varphi)$ on the Bloch sphere. $\\theta=0$ is $|0\\rangle$ (north pole), $\\theta=\\pi$ is
    $|1\\rangle$ (south pole).
    """
    return np.array([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])


def expectation(A, psi):
    """The expectation value $\\langle\\psi|A|\\psi\\rangle$ (§6.5), via ``numpy.vdot`` (real, Hermitian $A$)."""
    return np.vdot(psi, A @ psi).real


def bloch_vector(psi):
    """The Bloch vector $(\\langle\\sigma_x\\rangle,\\langle\\sigma_y\\rangle,\\langle\\sigma_z\\rangle)$ {eq}`eq-bloch`.

    The three Pauli expectation values, which are the Cartesian coordinates of the qubit's point on the
    Bloch sphere. A pure state has unit length; a global phase leaves it unchanged (the sphere is the
    space of rays, §6.1).
    """
    return np.array(
        [
            expectation(SIGMA_X, psi),
            expectation(SIGMA_Y, psi),
            expectation(SIGMA_Z, psi),
        ]
    )


def reduced_density_matrix(state4):
    """The reduced density matrix $\\rho_A=\\mathrm{Tr}_B|\\psi\\rangle\\langle\\psi|$ of qubit $A$ {eq}`eq-reduced`.

    Reshape the 4-component two-qubit state to the $2\\times2$ amplitude matrix $M_{ab}$ (rows = qubit
    $A$, columns = qubit $B$) and contract over qubit $B$: $\\rho_A=MM^{\\dagger}$. For an entangled
    pure state this comes out **mixed** — the diagnostic of entanglement.

    Parameters
    ----------
    state4 : numpy.ndarray
        A normalized two-qubit state (length 4), ordered $|q_Aq_B\\rangle=$ index $2q_A+q_B$.

    Returns
    -------
    numpy.ndarray
        The $2\\times2$ reduced density matrix of qubit $A$.
    """
    M = state4.reshape(2, 2)
    return M @ M.conj().T


def schmidt(state4):
    """The Schmidt coefficients of a two-qubit state {eq}`eq-entanglement`.

    The singular values of the reshaped $2\\times2$ amplitude matrix (`numpy.linalg.svd` with
    ``compute_uv=False``). The **Schmidt rank** is the number of nonzero coefficients: rank 1 means a
    product state, rank $\\ge2$ means entangled.
    """
    return np.linalg.svd(state4.reshape(2, 2), compute_uv=False)


def purity(rho):
    """The purity $\\mathrm{Tr}(\\rho^2)$ of a density matrix {eq}`eq-reduced`.

    One for a pure state, $\\tfrac1d$ for the maximally mixed state ($\\tfrac12$ for a qubit). Computed
    as ``numpy.trace(rho @ rho).real``.
    """
    return np.trace(rho @ rho).real


def entanglement_entropy(rho):
    """The entanglement entropy $S=-\\mathrm{Tr}(\\rho\\log_2\\rho)$ in bits {eq}`eq-reduced`.

    The 5.x Shannon/von-Neumann entropy of the reduced state's eigenvalues (`numpy.linalg.eigvalsh`),
    in base 2. Zero for a pure reduced state, one bit for a maximally mixed qubit — the amount of
    entanglement between the two halves.
    """
    eigenvalues = np.linalg.eigvalsh(rho)
    eigenvalues = eigenvalues[eigenvalues > 1e-12]
    return float(-np.sum(eigenvalues * np.log2(eigenvalues)))


def _bloch_frame(ax):
    """Draw the Bloch-sphere frame (wireframe + x, y, z axes) on a 3D axes — a plotting helper."""
    u = np.linspace(0, 2 * np.pi, 32)
    v = np.linspace(0, np.pi, 18)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(xs, ys, zs, color=SOFT, alpha=0.20, lw=0.4)
    for vec, lab in [((1.45, 0, 0), "x"), ((0, 1.45, 0), "y"), ((0, 0, 1.45), "z")]:
        ax.quiver(0, 0, 0, *vec, color=SOFT, lw=1.0, arrow_length_ratio=0.06)
        ax.text(
            vec[0] * 1.12,
            vec[1] * 1.12,
            vec[2] * 1.12,
            f"${lab}$",
            color=SOFT,
            fontsize=9,
        )
    ax.set_box_aspect([1, 1, 1])
    ax.set_axis_off()

## Exercise 1 — The Bloch sphere map

Map qubit states to points on the unit sphere and verify the map: the Bloch vector
$(\langle\sigma_x\rangle,\langle\sigma_y\rangle,\langle\sigma_z\rangle)$ of
$|\psi\rangle=\cos\tfrac \theta2|0\rangle+e^{i\varphi}\sin\tfrac\theta2|1\rangle$ equals
$(\sin\theta\cos\varphi,\sin\theta\sin \varphi,\cos\theta)$, and a pure state lands on the surface
($|\mathbf b|=1$) {eq}`eq-bloch`.

1. Build $|\psi\rangle$ with the `qubit` helper for a chosen $(\theta,\varphi)$.
2. Compute its Bloch vector with `bloch_vector` (the three `numpy.vdot`-based Pauli expectations).
3. Confirm it equals $(\sin\theta\cos\varphi,\sin\theta\sin\varphi,\cos\theta)$ with
   `numpy.allclose`, and that $|\mathbf b|=1$ (`numpy.linalg.norm`).
4. Check the special points: $|0\rangle\to(0,0,1)$, $|1\rangle\to(0,0,-1)$,
   $|{+}\rangle\to(1,0,0)$, $|{+}i\rangle\to(0,1,0)$. Frame: the ray becomes a point.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    b,
    expected,
    "the Bloch vector is the triple of Pauli expectation values; a pure qubit is a point on the unit sphere",
    rtol=1e-12,
)
validate.close(
    np.linalg.norm(b),
    1.0,
    "a pure state has a unit Bloch vector (it lies on the surface of the sphere)",
    rtol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — Global phase and the sphere

Show that the global-phase ambiguity of [§6.1](complex-vector-spaces.ipynb) is exactly the quotient that turns a state vector into
a point: multiplying $|\psi\rangle$ by $e^{i\alpha}$ leaves its Bloch vector unchanged, so the
sphere is the space of physical states (rays) {eq}`eq-bloch`.

1. Take a state $|\psi\rangle$ and the phase-shifted $e^{i\alpha}|\psi\rangle$
   (`numpy.exp(1j*alpha)`).
2. Compute both Bloch vectors with `bloch_vector`.
3. Confirm they are identical (`numpy.allclose`).
4. Read the meaning: the Bloch sphere *is* the set of rays — the "a global phase means nothing"
   of [§6.1](complex-vector-spaces.ipynb) made geometric.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    b_phased,
    b_plain,
    "a global phase leaves the Bloch point fixed — the sphere is the space of physical states (rays), §6.1 made geometric",
    rtol=1e-12,
)

## Exercise 3 — Orthogonality and evolution as geometry

Show two geometric facts: orthogonal qubit states are **antipodal** on the sphere, and Larmor
precession ([§6.7](time-evolution.ipynb)) is literally a **rotation** of the Bloch vector about the field axis
{eq}`eq-bloch-geometry`.

1. Confirm $|0\rangle$ and $|1\rangle$ (and another orthogonal pair) map to opposite points,
   $\mathbf b_\phi=-\mathbf b_\psi$ — noting the factor of two between the Hilbert-space angle
   ($90^\circ$) and the Bloch angle ($180^\circ$), the spin-$\tfrac12$ double cover.
2. Evolve $|{+} \rangle$ under the Larmor Hamiltonian $H=-\tfrac\omega2\sigma_z$ ([§6.7](time-evolution.ipynb)) using the
   spectral construction.
3. Compute the Bloch vector versus time.
4. Confirm it rotates about $z$ at frequency $\omega$: $\mathbf b(t)=(\cos\omega t,-\sin\omega
   t,0)$, the latitude fixed. Frame: Movement I's dynamics made visible.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    antipodal,
    "orthogonal qubit states are antipodal on the Bloch sphere (the factor-of-two double cover)",
)
validate.close(
    bloch_traj,
    predicted,
    "Larmor precession is a rotation of the Bloch vector about the field axis at frequency ω",
    atol=1e-10,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The tensor product and composite systems

Build two-qubit states and operators, and show how a single-qubit observable acts on the joint
system. Two qubits live in $\mathbb{C}^2\otimes\mathbb{C}^2=\mathbb{C}^4$; a product state is
$|a\rangle\otimes|b\rangle$ and a single-qubit observable is $A\otimes I$ {eq}`eq-tensor`.

1. Form a product state $|a\rangle\otimes|b\rangle$ with `numpy.kron` and confirm the basis
   ordering $\{|00\rangle,|01\rangle,|10\rangle,|11\rangle\}$.
2. Build the single-qubit observables $\sigma_z\otimes I=$`numpy.kron(SIGMA_Z, ID2)` and
   $I\otimes\sigma_z$.
3. Verify that $\langle\sigma_z\otimes I\rangle$ on $|a\rangle\otimes|b\rangle$ reports qubit
   $A$'s own $\langle \sigma_z\rangle$ (the `expectation` helper), and likewise $I\otimes\sigma_z$
   for qubit $B$.
4. Confirm a product state factorizes (its Schmidt rank, anticipating Exercise 5, is 1). Frame:
   how quantum mechanics describes more than one particle.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    [qubitA_sz, qubitB_sz],
    [expectation(SIGMA_Z, a), expectation(SIGMA_Z, b)],
    "two qubits live in the tensor product ℂ²⊗ℂ²; a single-qubit observable A⊗I reports that qubit's own expectation value",
    rtol=1e-12,
)

## Exercise 5 — Entanglement and the Schmidt decomposition

Determine whether a two-qubit state is a **product** or **entangled** using the Schmidt
decomposition. Apply it to the product state of Exercise 4 and to the Bell state
$|\Phi^+\rangle=(|00 \rangle+|11\rangle)/\sqrt2$, finding Schmidt rank 1 versus 2
{eq}`eq-entanglement`.

1. Reshape the 4-component state to a $2\times2$ amplitude matrix (`state4.reshape(2, 2)`).
2. Take its singular values with `numpy.linalg.svd(..., compute_uv=False)` — the Schmidt
   coefficients (the `schmidt` helper).
3. Count the nonzero coefficients: a **product** state has rank 1, an **entangled** state rank
   $\ge2$.
4. Apply to the product state and the Bell state — find rank 1 (coefficients $1,0$) versus rank 2
   (equal coefficients $1/\sqrt2,1/\sqrt2$). Frame: entanglement is non-factorizability.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    rank_product == 1 and rank_bell == 2,
    "an entangled state cannot be factored: the product state has Schmidt rank 1, the Bell state rank 2",
)

## Exercise 6 — The reduced density matrix: the part is mixed

Compute the reduced state of one qubit for the product state and for the Bell state, and quantify
its mixedness. The signature of entanglement is that the reduced state of a part is **mixed** even
though the whole is pure: for the Bell state $\rho_A$ is *maximally* mixed (purity $\tfrac12$,
entanglement entropy $1$ bit), while a product state gives a pure reduced state {eq}`eq-reduced`.

1. Form $\rho_A=\mathrm{Tr}_B|\psi\rangle\langle\psi|$ with the `reduced_density_matrix` helper
   (reshape to $M_{ab}$, contract over $B$: $\rho_A=MM^{\dagger}$).
2. Compute its eigenvalues (`numpy.linalg.eigvalsh`), purity $\mathrm{Tr}(\rho_A^2)$ (the `purity`
   helper), and entanglement entropy $S=-\sum\lambda\log_2\lambda$ (the `entanglement_entropy`
   helper).
3. For the **product** state find a pure $\rho_A$ (eigenvalues $0,1$; purity $1$; $S=0$).
4. For the **Bell** state find a maximally mixed $\rho_A$ (eigenvalues $\tfrac12,\tfrac12$; purity
   $\tfrac12$; $S=1$ bit) — its Bloch vector is the *origin*. Frame: the whole is sharp, the parts
   blurred; the information is in the correlations.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    [purity(rho_bell), entanglement_entropy(rho_bell)],
    [0.5, 1.0],
    "for the Bell state the reduced single-qubit state is maximally mixed (purity ½, entanglement entropy 1 bit) — the part is mixed though the whole is pure",
    rtol=1e-10,
)
validate.close(
    [purity(rho_product), entanglement_entropy(rho_product)],
    [1.0, 0.0],
    "for a product state the reduced state is pure (purity 1, entropy 0) — no entanglement",
    atol=1e-10,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Bell correlations: a puzzle *(student)*

In the Bell state, compute the joint correlations $\langle\sigma_z\otimes\sigma_z \rangle$ and
$\langle\sigma_x\otimes\sigma_x\rangle$ and the single-qubit averages $\langle\sigma_z \otimes
I\rangle$, and articulate why no pre-assigned local values can reproduce them: the two qubits are
perfectly correlated in *two incompatible bases* while each alone is random {eq}`eq-bell-taste`.

1. Build the joint observables $\sigma_z\otimes\sigma_z$, $\sigma_x\otimes\sigma_x$, and
   $\sigma_z\otimes I$ with `numpy.kron`.
2. Compute $\langle\sigma_z\otimes\sigma_z\rangle=+1$ (perfect $z$-correlation — same outcome both
   qubits) and $\langle\sigma_x\otimes\sigma_x\rangle=+1$ (perfect $x$-correlation) with the
   `expectation` helper.
3. Compute $\langle\sigma_z\otimes I\rangle =\langle I\otimes\sigma_z\rangle=0$ (each qubit,
   alone, perfectly random).
4. Argue the puzzle: if each qubit secretly carried definite $z$- *and* $x$-values fixed in
   advance, those values would have to be simultaneously perfectly correlated *and* locally random
   in two bases that cannot both be sharp — impossible for pre-assigned numbers. This is made
   quantitative by Bell's inequality ([§6.25](bell-inequality.ipynb)). Frame: the puzzle that breaks local realism.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    [corr_zz, corr_xx, marg_z],
    [1.0, 1.0, 0.0],
    "the Bell state is perfectly correlated in two incompatible bases (⟨σzσz⟩=⟨σxσx⟩=1) with random marginals (⟨σz⊗I⟩=0) — no local hidden variables can reproduce this",
    rtol=1e-12,
)

## Exercise 8 — The geometry of one qubit, the mystery of two

A single qubit is a **point on a sphere**: its state a ray turned into a location, its global phase
invisible, its evolution a rigid rotation, its measurements projections onto axes. That one picture
holds the whole of Movement I — superposition, measurement, uncertainty, and dynamics, all visible at
once. But **two** qubits do something the sphere cannot capture. They **entangle**, and then the joint
state is perfectly definite while each part is maximally uncertain, the reduced state sitting at the
dead center of its sphere, all the information hidden in correlations that are perfect in bases that
cannot both be sharp. This is where the classical picture fails completely and finally.

**This exercise (synthesis).** No new computation: the contrast is the result. We have spent five
notebooks in two dimensions and found there, in miniature, everything quantum mechanics has to say —
superposition ([§6.4](stern-gerlach-qubit.ipynb)), measurement ([§6.5](postulates.ipynb)), uncertainty ([§6.6](pauli-uncertainty.ipynb)), dynamics ([§6.7](time-evolution.ipynb)), and now entanglement.
Movement I is complete: we built the formalism and met its strangest consequence. What remains is
mostly a change of *arena*. Movement II ([§6.9](position-representation.ipynb)) takes the same formalism to **infinite** dimensions,
where the qubit's two amplitudes swell into a continuum — a **wave function** $\psi(x)$ — and the
spectral theorem we have leaned on throughout becomes the Schrödinger differential equation.

## Notebook summary

The geometry of one qubit and the mystery of two — the capstone of Movement I.

- **The Bloch sphere** {eq}`eq-bloch`: a qubit state is a point, its coordinates the Pauli
  expectations $(\langle\sigma_x\rangle,\langle\sigma_y\rangle,\langle\sigma_z\rangle)$; a pure state
  has $|\mathbf b|=1$.
- **Geometry** {eq}`eq-bloch-geometry`: a global phase is invisible (the sphere is the space of rays),
  orthogonal states are antipodal (the factor-of-two double cover), and unitary evolution is a
  rotation — Larmor precession made literal.
- **The tensor product** {eq}`eq-tensor`: two qubits live in $\mathbb{C}^2\otimes\mathbb{C}^2$
  (`numpy.kron`); a single-qubit observable is $A\otimes I$.
- **Entanglement** {eq}`eq-entanglement`: the Bell state is not a product — Schmidt rank 2
  (`numpy.linalg.svd`) versus a product's rank 1.
- **The reduced state** {eq}`eq-reduced`: $\rho_A=\mathrm{Tr}_B|\psi\rangle\langle\psi|$ is *maximally
  mixed* for the Bell state (purity $\tfrac12$, $S=1$ bit) though the whole is pure — the part is
  blurred, the information is in the correlations.
- **Bell correlations** {eq}`eq-bell-taste`: perfect in two incompatible bases ($\langle\sigma_z
  \sigma_z\rangle=\langle\sigma_x\sigma_x\rangle=1$) with random marginals — no local hidden variables
  ([§6.25](bell-inequality.ipynb)).

Movement I is complete. In two dimensions we found everything quantum mechanics has to say in
miniature; Movement II changes the arena, from two amplitudes to a continuum.

## Outlook

- **Bell's inequality made quantitative ([§6.25](bell-inequality.ipynb))**: the computational demonstration of a quantum
  violation of local realism.
- **The density matrix and mixed states ([§6.26](density-matrix.ipynb))**: states *inside* the Bloch ball; classical ignorance
  versus entanglement; the thermal density matrix ($\to$ Volume VII).
- **Wave mechanics (Movement II, [§6.9](position-representation.ipynb))**: the same formalism in infinite dimensions — the qubit's two
  amplitudes become a wave function $\psi(x)$.
- **The many-body tensor product and identical particles ([§6.20](identical-particles.ipynb))**.
- **Cross-reference** [§6.1](complex-vector-spaces.ipynb) (rays / global phase), [§6.6](pauli-uncertainty.ipynb) (Pauli expectations), [§6.7](time-evolution.ipynb) (precession), 5.x
  (entropy), and forward to [§6.9](position-representation.ipynb), [§6.25](bell-inequality.ipynb), [§6.26](density-matrix.ipynb), [§6.20](identical-particles.ipynb).

In [ ]:
from ecp.style import footer

footer()